In [ ]:
"""
LINEAGE BASED HYPERBOLIC ODE INTERPOLATOR FOR C. ELEGANS CONNECTOME
=======================================================

Architecture Overview
---------------------
We learn a continuous-time dynamical system on the Poincaré ball that
describes how neurons move through hyperbolic space as the worm develops.

The velocity field has two components:

  dz/dt = v_mech(z, k)  +  v_neural(z, h_GAT, h_lineage, log_k, t)
          ─────────────     ─────────────────────────────────────────
          mechanistic          neural residual
          (from radial         (learns everything the
          ODE regression)       mechanistic part misses)

Mechanistic component (from our earlier ODE regression):
  Fitted: dr/dt = 0.639 - 0.493·r - 0.085·log(k)
  In 2D:  v_mech = (dr/dt) · (z/‖z‖)     [radial force toward equilibrium]

Neural residual inputs per neuron:
  • z_flat   (2D) — current position via logmap₀, flattened to tangent space
  • h_GAT   (16D) — graph-aware neighbourhood features from frozen GAT
  • lin_dir  (2D) — unit direction toward lineage centroid (hyperedge pull)
  • lin_dist (1D) — distance to lineage centroid (how far from the group)
  • log_k    (1D) — log degree (connectivity context)
  • t        (1D) — normalised time in [0,1]
  Total:    23D  →  64  →  32  →  2

Why ODE?
  The 8 developmental stages are SNAPSHOTS of a continuous biological
  process. An ODE lets us:
    1. Interpolate to ANY time point, not just integer stages
    2. Enforce continuity of trajectories
    3. Learn a DYNAMICS MODEL, not just a mapping

Why lineage direction?
  From Level 1+2 analysis: neurons move COHERENTLY with their lineage
  group. The centroid direction encodes the "collective pull" — neurons
  that are far from their group centroid should experience a restoring
  force. This is the hyperedge contribution your mentor asked for.

Validation:
  Held-out stage = 4
  Train on transitions: 1→2, 2→3, 5→6, 6→7, 7→8
  Predict: start from stage 3, integrate ODE to t=1 → predicted stage 4
  Compare vs true stage 4: TP / FP / FN / AUC / F1 / Precision / Recall
  Baseline comparisons: geodesic midpoint, linear interpolation
"""

# ─────────────────────────────────────────────────────────────
# SECTION 0 — INSTALLS AND IMPORTS
# ─────────────────────────────────────────────────────────────
import subprocess, sys
def install(p):
    subprocess.check_call([sys.executable,"-m","pip","install",p,"-q"])
for p in ["geoopt","networkx","openpyxl","scipy","torchdiffeq"]:
    install(p)

import os, re, random, warnings
import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import geoopt
from torchdiffeq import odeint
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import roc_auc_score
from scipy import stats
from collections import Counter

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MANIFOLD = geoopt.PoincareBall(c=1.0)          # used everywhere
print(f"Device: {DEVICE}")
print("="*65)

# ─────────────────────────────────────────────────────────────
# SECTION 1 — DATA LOADING
# ─────────────────────────────────────────────────────────────
print("SECTION 1: Loading Data")
print("-"*40)

# ── 1a. Lineage tree ──────────────────────────────────────────
df_lin = pd.read_csv("cells_birth_and_pos.csv")
tree       = nx.DiGraph()
embryo_pos = {}
birth_time = {}
for _, row in df_lin.iterrows():
    p  = str(row["Parent Cell"]).strip()
    d1 = str(row["Daughter 1"]).strip()
    d2 = str(row["Daughter 2"]).strip()
    tree.add_edge(p, d1)
    tree.add_edge(p, d2)
    embryo_pos[p] = (float(row["parent_x"]),
                     float(row["parent_y"]),
                     float(row["parent_z"]))
    birth_time[p] = float(row["Birth Time"])
print(f"  Lineage tree: {tree.number_of_nodes()} cells")

# ── 1b. Connectome graphs ─────────────────────────────────────
def load_stage(s):
    df = pd.read_excel(f"witvliet_2020_{s}.xlsx")
    G  = nx.DiGraph()
    for _, row in df.iterrows():
        pre, post, w = str(row["pre"]).strip(), str(row["post"]).strip(), float(row["synapses"])
        if G.has_edge(pre, post): G[pre][post]["weight"] += w
        else:                     G.add_edge(pre, post, weight=w)
    G = G.to_undirected()
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    return G

graphs = {s: load_stage(s) for s in range(1, 9)}
for s, G in graphs.items():
    print(f"  Stage {s}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# ─────────────────────────────────────────────────────────────
# SECTION 2 — LINEAGE ASSIGNMENT (FIXED VERSION)
# ─────────────────────────────────────────────────────────────
print("\nSECTION 2: Lineage Assignment")
print("-"*40)

FOUNDERS = {"ABa":"orange","ABp":"steelblue","MS":"green",
            "E":"red","C":"purple","P3":"brown"}
N_LINEAGES = len(FOUNDERS)
FOUNDER_LIST = list(FOUNDERS.keys())

def prefix_lineage(c):
    c = c.strip()
    if c.startswith("ABa") or c.startswith("ABal") or c.startswith("ABar"): return "ABa"
    if c.startswith("ABp") or c.startswith("ABpl") or c.startswith("ABpr"): return "ABp"
    if c.startswith("MS"):  return "MS"
    if c.startswith("Ea") or c.startswith("Ep") or c.startswith("int"): return "E"
    if c.startswith("Ca") or c.startswith("Cp") or c=="C": return "C"
    if (c.startswith("Da") or c.startswith("Dp") or
        c.startswith("P3") or c.startswith("P4")): return "P3"
    return None

def build_csv_map(df_lin):
    m = {}
    for _, row in df_lin.iterrows():
        parent = str(row["Parent Cell"]).strip()
        pl     = prefix_lineage(parent)
        if pl is None: continue
        for d in [str(row["Daughter 1"]).strip(), str(row["Daughter 2"]).strip()]:
            if d and d not in m: m[d] = pl
    return m

csv_lineage_map = build_csv_map(df_lin)

# WormBase curated overrides (key neurons)
WORMBASE = {
    **{n:"ABa" for n in ["ADAL","ADAR","ADEL","ADER","AFDL","AFDR","AIML","AIMR",
       "AINL","AINR","AIYL","AIYR","AIZL","AIZR","ALA","ALML","ALMR","ALNL","ALNR",
       "ASEL","ASER","ASGL","ASGR","ASHL","ASHR","ASIL","ASIR","ASJL","ASJR",
       "ASKL","ASKR","AUAL","AUAR","AVAL","AVAR","AVBL","AVBR","AVDL","AVDR",
       "AVEL","AVER","AVG","AVHL","AVHR","AVJL","AVJR","AVKL","AVKR","AVL",
       "AWAL","AWAR","AWBL","AWBR","AWCL","AWCR","BAGL","BAGR","CANL","CANR",
       "CEPDL","CEPDR","CEPVL","CEPVR","NSML","NSMR","RIAL","RIAR","RIBL","RIBR",
       "RICL","RICR","RID","RIGL","RIGR","RIML","RIMR","RIPL","RIPR","RIR",
       "SAADL","SAADR","SAAVL","SAAVR","SABVL","SABVR","URADL","URADR",
       "URAVL","URAVR","URBL","URBR","URXL","URXR",
       "URYDL","URYDR","URYVL","URYVR"]},
    **{n:"ABp" for n in ["AIBL","AIBR","AS1","AS2","AS3","AS4","AS5","AS6","AS7",
       "AS8","AS9","AS10","AS11","BDUL","BDUR",
       "DA1","DA2","DA3","DA4","DA5","DA6","DA7","DA8","DA9",
       "DB1","DB2","DB3","DB4","DB5","DB6","DB7",
       "DD1","DD2","DD3","DD4","DD5","DD6","DVA","DVB","FLPL","FLPR",
       "HSNL","HSNR","IL1DL","IL1DR","IL1L","IL1R","IL1VL","IL1VR",
       "IL2DL","IL2DR","IL2L","IL2R","IL2VL","IL2VR","LUAL","LUAR",
       "OLLL","OLLR","OLQDL","OLQDR","OLQVL","OLQVR",
       "PHAL","PHAR","PHBL","PHBR","PHCL","PHCR","PLML","PLMR","PLNL","PLNR",
       "PQR","PVDL","PVDR","PVML","PVMR","PVN","PVQL","PVQR","PVT","PVWL","PVWR",
       "RIS","RMDL","RMDR","RMDVL","RMDVR","RMEL","RMER","RMED","RMEV",
       "RMFL","RMFR","RMGL","RMGR","RMHL","RMHR","SDQL","SDQR",
       "SIADL","SIADR","SIAVL","SIAVR","SIBDL","SIBDR","SIBVL","SIBVR",
       "SMBDL","SMBDR","SMBVL","SMBVR","SMDDL","SMDDR","SMDVL","SMDVR",
       "AVM","PVM","VA1","VA2","VA3","VA4","VA5","VA6","VA7","VA8","VA9",
       "VA10","VA11","VA12","VB1","VB2","VB3","VB4","VB5","VB6","VB7",
       "VB8","VB9","VB10","VB11","VC1","VC2","VC3","VC4","VC5","VC6",
       "VD1","VD2","VD3","VD4","VD5","VD6","VD7","VD8","VD9","VD10",
       "VD11","VD12","VD13","QL","QR","XXXL","XXXR","ADFL","ADFR",
       "AIAL","AIAR"]},
    **{n:"MS" for n in ["GLRDL","GLRDR","GLRL","GLRVL","GLRVR",
       "I1","I2","I3","I4","I5","I6","M1","M2","M3","M4","M5","MC","MI"]},
    "DVC":"C","PVR":"C",
}

def assign_lineage(neuron):
    if neuron in WORMBASE:     return WORMBASE[neuron]
    if neuron in csv_lineage_map: return csv_lineage_map[neuron]
    p = prefix_lineage(neuron)
    return p if p else "Unknown"

all_neurons = set()
for G in graphs.values(): all_neurons.update(G.nodes())
neuron_lineage = {n: assign_lineage(n) for n in all_neurons}
counts = Counter(neuron_lineage.values())
print("  Lineage assignments:")
for f in FOUNDER_LIST: print(f"    {f:5s}: {counts.get(f,0):3d}")
print(f"    {'Unknown':5s}: {counts.get('Unknown',0):3d}")


# ─────────────────────────────────────────────────────────────
# SECTION 3 — NODE FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────
def extract_features(G):
    """
    4 normalised features per neuron:
      [0] degree / max_degree
      [1] log(degree+1) / log(max_degree+1)
      [2] clustering coefficient
      [3] betweenness centrality / max_betweenness
    """
    nodes = list(G.nodes())
    N = len(nodes)
    deg     = dict(G.degree())
    clust   = nx.clustering(G)
    between = nx.betweenness_centrality(G, normalized=True, k=min(100, N))
    md  = max(deg.values())    + 1e-9
    mb  = max(between.values())+ 1e-9
    F = np.array([[deg[n]/md,
                   np.log(deg[n]+1)/np.log(md+1),
                   clust[n],
                   between[n]/mb] for n in nodes], dtype=np.float32)
    return nodes, F

stage_features = {}
for s in range(1, 9):
    nodes, feat = extract_features(graphs[s])
    stage_features[s] = {"nodes": nodes, "feat": feat}

# ─────────────────────────────────────────────────────────────
# SECTION 4 — GRAPH ATTENTION NETWORK
# ─────────────────────────────────────────────────────────────
#
# Two-layer GAT enriches each neuron's representation using its
# neighbourhood structure. After enrichment, the 16-dim vector h_i
# encodes "what kind of connectivity pattern does this neuron have?"
#
# Attention formula (Veličković et al. 2018):
#   e_ij = LeakyReLU( a^T [Wh_i ‖ Wh_j] )
#   α_ij = softmax_j(e_ij)
#   h_i' = ELU( Σ_{j∈N(i)} α_ij · Wh_j )

class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.W   = nn.Linear(in_dim, out_dim, bias=False)
        self.a   = nn.Linear(2*out_dim, 1, bias=False)
        self.lr  = nn.LeakyReLU(0.2)
        self.dp  = nn.Dropout(dropout)
    def forward(self, X, A):
        N = X.size(0)
        Wh = self.W(X)
        ei = Wh.unsqueeze(1).expand(N,N,-1)
        ej = Wh.unsqueeze(0).expand(N,N,-1)
        e  = self.lr(self.a(torch.cat([ei,ej],dim=-1)).squeeze(-1))
        e  = e.masked_fill(A==0, float("-inf"))
        al = F.softmax(e, dim=1)
        al = torch.where(torch.isnan(al), torch.zeros_like(al), al)
        al = self.dp(al)
        return F.elu(torch.mm(al, Wh))

class GATEncoder(nn.Module):
    def __init__(self, in_dim=4, hidden=32, out_dim=16):
        super().__init__()
        self.l1 = GATLayer(in_dim, hidden)
        self.l2 = GATLayer(hidden, out_dim)
    def forward(self, X, A):
        return self.l2(self.l1(X, A), A)

# ─────────────────────────────────────────────────────────────
# SECTION 5 — HYPERBOLIC EMBEDDER
# ─────────────────────────────────────────────────────────────
# Poincaré ball with Fermi-Dirac decoder.
# Fermi-Dirac:  P(edge|i,j) = 1/(exp((d_H - r)/T) + 1)
# Loss: weighted BCE (handles 94% non-edge class imbalance)

class HyperbolicEmbedder(nn.Module):
    def __init__(self, n, c=1.0):
        super().__init__()
        self.manifold = geoopt.PoincareBall(c=c)
        self.z = geoopt.ManifoldParameter(
            torch.randn(n, 2)*0.01, manifold=self.manifold)
        self.r = nn.Parameter(torch.tensor(0.5))
        self.T = nn.Parameter(torch.tensor(0.3))
    def forward(self):
        D = self.manifold.dist(self.z.unsqueeze(1), self.z.unsqueeze(0))
        P = 1.0/(torch.exp((D - self.r)/self.T.clamp(0.05)) + 1.0)
        return P, D
    def radii(self):
        return torch.norm(self.z.detach(), dim=1)

def build_adj(G, nodes):
    n = len(nodes); idx = {v:i for i,v in enumerate(nodes)}
    A = np.zeros((n,n), dtype=np.float32)
    for u,v in G.edges():
        if u in idx and v in idx: A[idx[u],idx[v]] = A[idx[v],idx[u]] = 1.0
    return torch.tensor(A)

def procrustes_align(z_src, z_tgt):
    M = z_src.T @ z_tgt; U,_,Vt = torch.linalg.svd(M)
    d = torch.det(Vt.T @ U.T); D = torch.diag(torch.tensor([1.,d.item()]))
    return z_src @ (Vt.T @ D @ U.T).T

def train_stage(G, nodes, epochs=600, z_init=None):
    n = len(nodes); A = build_adj(G, nodes).to(DEVICE)
    emb = HyperbolicEmbedder(n).to(DEVICE)
    if z_init is not None:
        with torch.no_grad(): emb.z.data = emb.manifold.projx(z_init.to(DEVICE))
    n_pos = A.sum().clamp(1); n_neg = A.numel()-n_pos
    pw    = (n_neg/n_pos).clamp(max=5.).item()
    opt   = geoopt.optim.RiemannianAdam(
        [{"params":[emb.z]},{"params":[emb.r,emb.T],"lr":0.005}], lr=0.01)
    emb.train()
    for ep in range(epochs):
        opt.zero_grad()
        P,D = emb()
        Ps  = P.clamp(1e-6,1-1e-6)
        wm  = torch.where(A==1, torch.full_like(A,pw), torch.ones_like(A))
        rep = F.relu(0.5 - D[A==0]).mean()
        loss= F.binary_cross_entropy(Ps,A,weight=wm) + 0.2*rep
        loss.backward(); opt.step()
    emb.eval(); return emb


# ─────────────────────────────────────────────────────────────
# SECTION 6 — PRE-TRAIN GAT + TRAIN ALL EMBEDDINGS
# ─────────────────────────────────────────────────────────────
HELD_OUT = 4
TRAIN_STAGES = [s for s in range(1,9) if s != HELD_OUT]

print(f"\nSECTION 6: Pre-training GAT")
print(f"  Held-out stage: {HELD_OUT}")
print(f"  Training stages: {TRAIN_STAGES}")

gat = GATEncoder(4, 32, 16).to(DEVICE)
gat_opt = optim.Adam(gat.parameters(), lr=0.001, weight_decay=1e-4)

for ep in range(700):
    gat.train(); ep_loss = 0.
    for s in TRAIN_STAGES:
        nodes = stage_features[s]["nodes"]
        feat  = stage_features[s]["feat"]
        A     = build_adj(graphs[s], nodes).to(DEVICE)
        X     = torch.tensor(feat).to(DEVICE)
        gat_opt.zero_grad()
        h     = gat(X, A)
        hn    = F.normalize(h, dim=1)
        sim   = ((hn @ hn.T)+1)/2
        n_pos = A.sum().clamp(1); n_neg = A.numel()-n_pos
        pw    = (n_neg/n_pos).clamp(max=5.).item()
        wm    = torch.where(A==1, torch.full_like(A,pw), torch.ones_like(A))
        loss  = F.binary_cross_entropy(sim.clamp(1e-6,1-1e-6), A, weight=wm)
        loss.backward(); gat_opt.step(); ep_loss += loss.item()
    if ep % 100 == 0: print(f"  GAT ep {ep:3d}  loss {ep_loss/len(TRAIN_STAGES):.4f}")

for p in gat.parameters(): p.requires_grad_(False)
gat.eval(); print("  GAT frozen.")

print(f"\nTraining hyperbolic embeddings …")
stage_embs  = {}
stage_nodes = {}
prev_emb, prev_nodes = None, None

for s in range(1, 9):
    print(f"  Stage {s} …", end=" ", flush=True)
    G, nodes = graphs[s], stage_features[s]["nodes"]
    feat     = stage_features[s]["feat"]
    z_init   = None
    if prev_emb is not None:
        zw     = torch.randn(len(nodes),2)*0.01
        common = [v for v in nodes if v in prev_nodes]
        if len(common)>3:
            ip = [prev_nodes.index(v) for v in common]
            ic = [nodes.index(v)      for v in common]
            zw[ic] = prev_emb.z.detach().cpu()[ip]
        z_init = zw
    emb = train_stage(G, nodes, epochs=600, z_init=z_init)
    # Procrustes align to previous
    if prev_emb is not None:
        common = [v for v in nodes if v in prev_nodes]
        if len(common)>3:
            ip = [prev_nodes.index(v) for v in common]
            ic = [nodes.index(v)      for v in common]
            zc  = emb.z.detach().cpu()[ic]
            zp  = prev_emb.z.detach().cpu()[ip]
            M   = zc.T @ zp; U,_,Vt = torch.linalg.svd(M)
            d2  = torch.det(Vt.T @ U.T)
            D2  = torch.diag(torch.tensor([1., d2.item()]))
            R   = Vt.T @ D2 @ U.T
            z_rot = emb.z.detach().cpu() @ R.T
            with torch.no_grad():
                emb.z.data = emb.manifold.projx(z_rot.to(DEVICE))
    stage_embs[s]  = emb
    stage_nodes[s] = nodes
    prev_emb, prev_nodes = emb, nodes
    print(f"done | r={emb.r.item():.3f} T={emb.T.item():.3f}")


# ─────────────────────────────────────────────────────────────
# SECTION 7 — LINEAGE HYPEREDGE FEATURES
# ─────────────────────────────────────────────────────────────
# For each neuron at each stage compute:
#   lin_dir  (2D): unit vector FROM neuron TOWARD its lineage centroid
#                  computed in tangent space at the origin via logmap₀
#   lin_dist (1D): hyperbolic distance to lineage centroid
#
# These encode the "collective pull" — neurons far from their group
# centroid should feel a restoring force toward it.
# This is the hyperedge contribution: the ODE learns whether/how much
# the neuron should be pulled back toward its lineage group.

def einstein_midpoint(z_group, c=1.0):
    """Fréchet mean on Poincaré ball (exact for 2D)."""
    norms_sq = torch.sum(z_group**2, dim=1)
    lam = 2.0/(1.0 - c*norms_sq).clamp(min=1e-6)
    num = (lam.unsqueeze(1)*z_group).sum(0)
    den = lam.sum() - 1.0
    return MANIFOLD.projx((num/den.clamp(1e-6)).unsqueeze(0)).squeeze(0)

MAN_CPU = geoopt.PoincareBall(c=1.0)

def compute_lineage_features(nodes, z_cpu, neuron_lineage):
    """
    Returns lin_features: (N, 3) tensor
      col 0,1 : unit direction toward lineage centroid (in tangent space at origin)
      col 2   : hyperbolic distance to lineage centroid
    """
    idx  = {v:i for i,v in enumerate(nodes)}
    N    = len(nodes)
    lf   = torch.zeros(N, 3)

    # Compute centroid per group
    centroids = {}
    for f in FOUNDER_LIST:
        members = [v for v in nodes if neuron_lineage.get(v)==f]
        if len(members) < 2: continue
        mi = [idx[v] for v in members]
        centroids[f] = einstein_midpoint(z_cpu[mi])

    for i, node in enumerate(nodes):
      f = neuron_lineage.get(node)
      if f not in centroids: continue
      c_hat = centroids[f].unsqueeze(0).cpu()   # ← force CPU
      z_i   = z_cpu[i].unsqueeze(0).cpu()       # ← force CPU
      dist  = MAN_CPU.dist(z_i, c_hat).item()
      c_flat = MAN_CPU.logmap0(c_hat).squeeze(0)
      z_flat = MAN_CPU.logmap0(z_i).squeeze(0)
      diff   = c_flat - z_flat
      norm   = diff.norm().clamp(min=1e-8)
      lf[i, 0] = diff[0]/norm
      lf[i, 1] = diff[1]/norm
      lf[i, 2] = dist                     # scalar distance
    return lf


# ─────────────────────────────────────────────────────────────
# SECTION 8 — HYBRID NEURAL ODE
# ─────────────────────────────────────────────────────────────
#
# The ODE operates in the TANGENT SPACE at the ORIGIN (logmap₀).
# This is the standard approach for neural ODEs on Poincaré balls:
#   1. logmap₀(z) maps the ball to flat R² (tangent at origin)
#   2. All neural network computations happen in R²
#   3. expmap₀(v) maps the velocity back to the ball
#
# Why tangent space at origin (not at z)?
#   Operating at z requires parallel transport of gradients at each
#   integration step — expensive and unstable. Origin tangent space
#   is a good approximation when ‖z‖ < 0.7 (true for most neurons).
#
# Mechanistic component (from fitted radial ODE, Section 25 of analysis):
#   Fitted ODE:  dr/dt = 0.639 - 0.493·r - 0.085·log(k)
#   This says neurons are pulled toward equilibrium radius
#   r* = (0.639 - 0.085·log(k)) / 0.493
#   High-degree hubs have low r* → pulled toward center
#   Low-degree periphery neurons have high r* → stay near boundary
#
# The full 2D mechanistic velocity:
#   v_mech = (dr/dt) · ẑ   where ẑ = z/‖z‖ (radial unit vector)

# Fitted parameters from regression (Section 25, R²=0.32)
A_MECH = 0.639
B_MECH = 0.493
C_MECH = 0.085

class HybridODEFunc(nn.Module):
    """
    Hybrid mechanistic + neural ODE velocity field.

    Stored context (set before each integration call):
      self.h_node   (N, 16)  — GAT features
      self.h_lin    (N,  3)  — lineage features [dir_x, dir_y, dist]
      self.log_k    (N,  1)  — log degree
    """
    def __init__(self, h_dim=16, lin_dim=3):
        super().__init__()
        # Input: z_flat(2) + h_node(16) + h_lin(3) + log_k(1) + t(1) = 23
        inp = 2 + h_dim + lin_dim + 1 + 1
        self.net = nn.Sequential(
            nn.Linear(inp, 64), nn.Tanh(),
            nn.Linear(64,  64), nn.Tanh(),
            nn.Linear(64,  32), nn.Tanh(),
            nn.Linear(32,   2),
        )
        # Learnable weight balancing mechanistic vs neural
        self.alpha = nn.Parameter(torch.tensor(0.5))

        # Context — set by set_context() before each call
        self.h_node = None
        self.h_lin  = None
        self.log_k  = None

    def set_context(self, h_node, h_lin, log_k):
        self.h_node = h_node   # (N, 16)
        self.h_lin  = h_lin    # (N,  3)
        self.log_k  = log_k    # (N,  1)

    def forward(self, t, z):
        """
        t : scalar tensor (current time in [0,1])
        z : (N, 2) positions on Poincaré ball
        returns: (N, 2) velocity in tangent space at origin
        """
        N = z.shape[0]
        dev = z.device

        # ── Map to tangent space at origin ─────────────────────
        # logmap₀(z) = (arctanh(√c·‖z‖)/(√c·‖z‖)) · z
        # This gives a flat representation of the positions
        man_dev = geoopt.PoincareBall(c=1.0)
        z_flat  = man_dev.logmap0(z)     # (N, 2)

        # ── Mechanistic component ───────────────────────────────
        # Radial velocity: dr/dt = a - b·r - c·log(k)
        r = z.norm(dim=1, keepdim=True).clamp(min=1e-8)   # (N,1)
        log_k = self.log_k.to(dev) if self.log_k is not None \
                else torch.zeros(N,1,device=dev)
        dr_dt = A_MECH - B_MECH*r - C_MECH*log_k   # (N,1)
        z_hat = z / r                                 # (N,2) radial unit vector
        v_mech = dr_dt * z_hat                        # (N,2)

        # ── Neural residual ─────────────────────────────────────
        t_vec = torch.ones(N, 1, device=dev) * t
        h_n   = self.h_node.to(dev) if self.h_node is not None \
                else torch.zeros(N,16,device=dev)
        h_l   = self.h_lin.to(dev) if self.h_lin is not None \
                else torch.zeros(N,3,device=dev)
        inp   = torch.cat([z_flat, h_n, h_l, log_k, t_vec], dim=1)
        v_res = self.net(inp)

        # ── Combine with learned weight α ───────────────────────
        α = torch.sigmoid(self.alpha)
        return α * v_mech + (1-α) * v_res


def integrate_ode(ode_func, z0, t_span, method="rk4"):
    """
    Integrate the ODE from t_span[0] to t_span[1].
    Uses rk4 (fixed step) which is faster than dopri5 for small N.
    Returns z at final time on the Poincaré ball.
    """
    t_eval = torch.linspace(t_span[0], t_span[1], 10).to(z0.device)
    z_traj = odeint(ode_func, z0, t_eval, method=method)
    z_final = z_traj[-1]
    man_dev = geoopt.PoincareBall(c=1.0)
    return man_dev.projx(z_final)


# ─────────────────────────────────────────────────────────────
# SECTION 9 — ODE TRAINING
# ─────────────────────────────────────────────────────────────
# Training objective (per transition s → s+1):
#
#   L = λ_emb · MSE(z_pred, z_true)
#     + λ_edge· BCE(P_pred, A_true)
#
# MSE is in LOGMAP₀ space (flat equivalent) to avoid manifold
# distance issues in the gradient computation.
#
# The BCE uses the Fermi-Dirac decoder with the LEARNED r and T
# from the target stage embedding. This ensures the edge loss
# uses the correct length scale for that stage.
#
# Training stages: all transitions NOT involving the held-out stage.
# For HELD_OUT=4: train on 1→2, 2→3, 5→6, 6→7, 7→8

ode_func  = HybridODEFunc().to(DEVICE)
ode_opt   = optim.Adam(ode_func.parameters(), lr=0.005)
scheduler = optim.lr_scheduler.StepLR(ode_opt, step_size=200, gamma=0.5)

# Build list of valid transitions (exclude held-out)
# Distance-weighted transitions: closer to held-out stage → higher weight
# Stage 4 is held out. Closest transitions are 2→3 and 5→6 (distance 1),
# then 1→2 and 6→7 (distance 2), then 7→8 (distance 3).
# We exclude early expansion phases (1→2, 2→3) from the transition set
# because their dynamics (spread explosion) are opposite to 3→4 (convergence).
# Keeping only post-peak transitions gives the ODE the right dynamics to learn.

transitions_weighted = [
    (2, 3, 1.0),   # closest before held-out: similar magnitude, expansion end
    (5, 6, 2.0),   # closest after held-out: same convergence direction as 3→4
    (6, 7, 2.0),   # same convergence phase
    (7, 8, 1.5),   # slightly further but same phase
]
# Note: removed (1,2) — pure expansion phase, misleads the ODE about 3→4 dynamics
print(f"  Weighted transitions: {[(a,b,w) for a,b,w in transitions_weighted]}")
print(f"\nSECTION 9: Training ODE Interpolator")
print(f"  Valid transitions: {[(a,b) for a,b,w in transitions_weighted]}")
print(f"  Epochs: 800")

λ_emb  = 1.0
λ_edge = 0.5

man_dev = geoopt.PoincareBall(c=1.0)

loss_history = []

for ep in range(800):
    ode_func.train(); ep_loss = 0.

    for (s, s_next, t_weight) in transitions_weighted:
        nodes_s    = stage_nodes[s]
        nodes_snext= stage_nodes[s_next]
        common     = [v for v in nodes_s if v in nodes_snext]
        if len(common) < 10: continue

        ip = [nodes_s.index(v)     for v in common]
        iq = [nodes_snext.index(v) for v in common]

        # ── Get embeddings ──────────────────────────────────────
        z_s    = stage_embs[s].z.detach()[ip]           # (M,2) on GPU
        z_next = stage_embs[s_next].z.detach()[iq]      # (M,2)

        # ── GAT features ────────────────────────────────────────
        G_s   = graphs[s]
        feat_s= torch.tensor(stage_features[s]["feat"]).to(DEVICE)
        A_s   = build_adj(G_s, nodes_s).to(DEVICE)
        with torch.no_grad():
            h_all = gat(feat_s, A_s)
        h_node = h_all[ip]                              # (M,16)

        # ── Lineage features ─────────────────────────────────────
        z_cpu_s = stage_embs[s].z.detach().cpu()
        lf_all  = compute_lineage_features(nodes_s, z_cpu_s, neuron_lineage)
        h_lin   = lf_all[ip].to(DEVICE)                # (M,3)

        # ── Log degree ───────────────────────────────────────────
        degs  = torch.tensor(
            [np.log(G_s.degree(v)+1) for v in common],
            dtype=torch.float32, device=DEVICE).unsqueeze(1)  # (M,1)

        # ── Set ODE context ──────────────────────────────────────
        ode_func.set_context(h_node, h_lin, degs)

        # ── Integrate ODE ────────────────────────────────────────
        t_span = torch.tensor([0., 1.], device=DEVICE)
        z_pred = integrate_ode(ode_func, z_s, t_span)  # (M,2)

        # ── Embedding loss (MSE in logmap₀ space) ────────────────
        z_pred_flat = man_dev.logmap0(z_pred)
        z_next_flat = man_dev.logmap0(z_next)
        loss_emb    = F.mse_loss(z_pred_flat, z_next_flat)

        # ── Edge reconstruction loss ──────────────────────────────
        # Build true adjacency for common nodes in next stage
        A_next_full = build_adj(graphs[s_next], nodes_snext).to(DEVICE)
        A_common    = A_next_full[torch.tensor(iq)][:,torch.tensor(iq)]
        M = len(common)
        man_dev_loop = geoopt.PoincareBall(c=1.0)
        D_pred = man_dev_loop.dist(z_pred.unsqueeze(1), z_pred.unsqueeze(0))
        # Use interpolated r/T instead of target stage's true values.
        # This makes training consistent with evaluation.
        r_fd = 0.5 * (stage_embs[s].r.item() + stage_embs[s_next].r.item())
        T_fd = 0.5 * (stage_embs[s].T.item() + stage_embs[s_next].T.item())
        P_pred = (1.0/(torch.exp((D_pred - r_fd)/max(T_fd,0.05)) + 1.0)).clamp(1e-6,1-1e-6)
        n_pos  = A_common.sum().clamp(1)
        n_neg  = A_common.numel() - n_pos
        pw     = (n_neg/n_pos).clamp(max=5.).item()
        wm     = torch.where(A_common==1, torch.full_like(A_common,pw), torch.ones_like(A_common))
        loss_edge = F.binary_cross_entropy(P_pred, A_common, weight=wm)

        # Sparsity penalty: penalise the mean predicted probability being too high.
        # Target sparsity = actual edge density of the training graph.
        edge_density = A_common.sum() / A_common.numel()
        mean_pred_prob = P_pred.mean()
        loss_sparse = F.mse_loss(mean_pred_prob,
                                  edge_density.detach().to(DEVICE))

        loss = t_weight * (λ_emb*loss_emb + λ_edge*loss_edge + 0.3*loss_sparse)
        ode_opt.zero_grad(); loss.backward(); ode_opt.step()
        ep_loss += loss.item()

    scheduler.step()
    loss_history.append(ep_loss/max(len(transitions_weighted),1))
    if ep % 100 == 0:
        print(f"  Epoch {ep:3d} | loss {ep_loss/max(len(transitions_weighted),1):.4f} "
              f"| lr {scheduler.get_last_lr()[0]:.5f} "
              f"| α={torch.sigmoid(ode_func.alpha).item():.3f}")

print("ODE training complete.")


# ─────────────────────────────────────────────────────────────
# SECTION 10 — PREDICTION AND BASELINES
# ─────────────────────────────────────────────────────────────
# Predict held-out stage 4 using three methods:
#
#   Method 1 — Geodesic Midpoint (baseline from Level 3 work)
#              z₄_pred = exp_{z₃}(0.5 · log_{z₃}(z₅))
#
#   Method 2 — Linear interpolation (flat Euclidean baseline)
#              z₄_pred = 0.5 · (z₃ + z₅)
#
#   Method 3 — Hybrid ODE (our model)
#              z₄_pred = ODE(z₃, context_3, t=0→1)

print(f"\nSECTION 10: Predicting Stage {HELD_OUT}")
print("-"*40)

# Common nodes across stages 3, 4, 5
nodes3     = stage_nodes[3]
nodes4     = stage_nodes[HELD_OUT]
nodes5     = stage_nodes[5]
common_345 = sorted(set(nodes3) & set(nodes4) & set(nodes5))
print(f"  Common nodes (stages 3,4,5): {len(common_345)}")

i3  = [nodes3.index(v) for v in common_345]
i4  = [nodes4.index(v) for v in common_345]
i5  = [nodes5.index(v) for v in common_345]

z3 = stage_embs[3].z.detach().cpu()[i3]
z5 = stage_embs[5].z.detach().cpu()[i5]
z4_true = stage_embs[HELD_OUT].z.detach().cpu()[i4]

# ── Method 1: Geodesic midpoint ───────────────────────────────
def geodesic_midpoint(za, zb, t=0.5):
    log_v = MAN_CPU.logmap(za, zb)
    return MAN_CPU.projx(MAN_CPU.expmap(za, t*log_v))

z4_geo = geodesic_midpoint(z3, z5, t=0.5)
print(f"  Geodesic midpoint: done")

# ── Method 2: Linear interpolation ────────────────────────────
z4_lin = 0.5*(z3 + z5)
z4_lin = MAN_CPU.projx(z4_lin)   # project onto ball
print(f"  Linear interpolation: done")

# ── Method 3: Hybrid ODE ──────────────────────────────────────
ode_func.eval()
with torch.no_grad():
    # Build context for stage 3 → stage 4 prediction
    G3       = graphs[3]
    feat3    = torch.tensor(stage_features[3]["feat"]).to(DEVICE)
    A3_full  = build_adj(G3, nodes3).to(DEVICE)
    h_all3   = gat(feat3, A3_full)
    h_node3  = h_all3[i3]

    z3_cpu   = stage_embs[3].z.detach().cpu()
    lf_all3  = compute_lineage_features(nodes3, z3_cpu, neuron_lineage)
    h_lin3   = lf_all3[i3].to(DEVICE)

    degs3    = torch.tensor(
        [np.log(G3.degree(v)+1) for v in common_345],
        dtype=torch.float32, device=DEVICE).unsqueeze(1)

    ode_func.set_context(h_node3, h_lin3, degs3)
    t_span   = torch.tensor([0., 1.], device=DEVICE)
    z4_ode   = integrate_ode(ode_func, z3.to(DEVICE), t_span).cpu()

print(f"  ODE prediction: done")


# ─────────────────────────────────────────────────────────────
# SECTION 11 — EVALUATION AND METRICS
# ─────────────────────────────────────────────────────────────
# We evaluate THREE aspects:
#
# A. EMBEDDING QUALITY
#    MSE between predicted and true positions (in logmap₀ space)
#
# B. EDGE PREDICTION (TP/FP/FN/AUC)
#    Decode edges from predicted positions via Fermi-Dirac,
#    compare against true stage 4 adjacency.
#    Scan thresholds to find optimal F1.
#
# C. BASELINE COMPARISON TABLE
#    Geodesic | Linear | Hybrid ODE

print(f"\nSECTION 11: Evaluation")
print("="*65)

# True stage 4 adjacency (restricted to common nodes)
A4_full   = build_adj(graphs[HELD_OUT], nodes4).numpy()
A4_common = A4_full[np.ix_(i4, i4)]

def evaluate_method(z_pred, A_true, label,
                    r_fd=None, T_fd=None):
    """Full evaluation: embedding MSE + edge metrics at best-F1 threshold."""
    N = z_pred.shape[0]
    # Embedding MSE
    z_true_cpu = z4_true
    z_pred_flat = MAN_CPU.logmap0(z_pred.float())
    z_true_flat = MAN_CPU.logmap0(z_true_cpu.float())
    emb_mse = F.mse_loss(z_pred_flat, z_true_flat).item()

    # Pairwise distances → edge probs
    D = MAN_CPU.dist(z_pred.unsqueeze(1), z_pred.unsqueeze(0)).detach().numpy()
    r = r_fd if r_fd else 1.0
    T = T_fd if T_fd else 0.3
    P = 1.0/(np.exp((D - r)/max(T, 0.05)) + 1.0)

    mask   = np.triu(np.ones((N,N),dtype=bool), k=1)
    y_true = A_true[mask].astype(int)
    y_prob = P[mask]

    auc = roc_auc_score(y_true, y_prob)

    # Scan thresholds for best F1
    best = {"f1": -1}
    for thr in np.arange(0.05, 0.95, 0.02):
        yp = (y_prob >= thr).astype(int)
        TP = int(((yp==1)&(y_true==1)).sum())
        FP = int(((yp==1)&(y_true==0)).sum())
        FN = int(((yp==0)&(y_true==1)).sum())
        TN = int(((yp==0)&(y_true==0)).sum())
        pr = TP/(TP+FP+1e-9); re = TP/(TP+FN+1e-9)
        f1 = 2*pr*re/(pr+re+1e-9)
        if f1 > best["f1"]:
            best = dict(thr=round(thr,2),TP=TP,FP=FP,FN=FN,TN=TN,
                        precision=pr,recall=re,f1=f1,auc=auc,
                        emb_mse=emb_mse,y_true=y_true,y_prob=y_prob,
                        D=D,P=P)
    print(f"\n  [{label}]")
    print(f"    Embedding MSE     : {emb_mse:.5f}")
    print(f"    AUC               : {best['auc']:.4f}")
    print(f"    Best threshold    : {best['thr']}")
    print(f"    True  Positives   : {best['TP']}")
    print(f"    False Positives   : {best['FP']}")
    print(f"    False Negatives   : {best['FN']}")
    print(f"    True  Negatives   : {best['TN']}")
    print(f"    Precision         : {best['precision']:.4f}")
    print(f"    Recall            : {best['recall']:.4f}")
    print(f"    F1 Score          : {best['f1']:.4f}")
    return best

# r and T for the held-out stage are UNKNOWN at prediction time.
# We interpolate them from the two bracketing stages, consistent with
# how we interpolate positions. Using the true stage 4 values was
# information leakage that made FP artificially high.
r_pred = 0.5 * (stage_embs[3].r.item() + stage_embs[5].r.item())
T_pred = 0.5 * (stage_embs[3].T.item() + stage_embs[5].T.item())
print(f"  Interpolated r={r_pred:.3f}, T={T_pred:.3f}")
print(f"  (True stage 4: r={stage_embs[HELD_OUT].r.item():.3f}, "
      f"T={stage_embs[HELD_OUT].T.item():.3f})")

res_geo = evaluate_method(z4_geo, A4_common, "Geodesic Midpoint",    r_pred, T_pred)
res_lin = evaluate_method(z4_lin, A4_common, "Linear Interpolation", r_pred, T_pred)
res_ode = evaluate_method(z4_ode, A4_common, "Hybrid ODE",           r_pred, T_pred)

# Summary comparison table
print("\n" + "="*65)
print(f"  SUMMARY TABLE — Stage {HELD_OUT} Prediction")
print("="*65)
print(f"  {'Metric':<22} {'Geodesic':>12} {'Linear':>12} {'Hybrid ODE':>12}")
print(f"  {'-'*58}")
for key, label in [("emb_mse","Embedding MSE"),("auc","AUC"),
                   ("TP","True Positives"),("FP","False Positives"),
                   ("FN","False Negatives"),("precision","Precision"),
                   ("recall","Recall"),("f1","F1 Score")]:
    gv = res_geo[key]; lv = res_lin[key]; ov = res_ode[key]
    if isinstance(gv, float):
        print(f"  {label:<22} {gv:>12.4f} {lv:>12.4f} {ov:>12.4f}")
    else:
        print(f"  {label:<22} {gv:>12d} {lv:>12d} {ov:>12d}")
print("="*65)


# ─────────────────────────────────────────────────────────────
# SECTION 12 — MULTI-STAGE EVALUATION (leave-one-out style)
# ─────────────────────────────────────────────────────────────
# Beyond the primary held-out test, we also evaluate how well
# the trained ODE predicts WITHIN-TRAINING stage transitions.
# This tells us whether the model has learned a general dynamical
# rule or just memorised specific transitions.

print(f"\nSECTION 12: Within-training transition quality")
print("-"*40)

within_results = []
ode_func.eval()
with torch.no_grad():
    for (s, s_next) in transitions:
        nodes_s     = stage_nodes[s]
        nodes_snext = stage_nodes[s_next]
        common      = [v for v in nodes_s if v in nodes_snext]
        if len(common) < 10: continue
        ip = [nodes_s.index(v)      for v in common]
        iq = [nodes_snext.index(v)  for v in common]

        z_s     = stage_embs[s].z.detach()[ip]
        z_next  = stage_embs[s_next].z.detach()[iq].cpu()

        feat_s  = torch.tensor(stage_features[s]["feat"]).to(DEVICE)
        A_s_dev = build_adj(graphs[s], nodes_s).to(DEVICE)
        h_all   = gat(feat_s, A_s_dev)
        h_n     = h_all[ip]
        lf_all  = compute_lineage_features(nodes_s,
                      stage_embs[s].z.detach().cpu(), neuron_lineage)
        h_l     = lf_all[ip].to(DEVICE)
        degs    = torch.tensor(
            [np.log(graphs[s].degree(v)+1) for v in common],
            dtype=torch.float32, device=DEVICE).unsqueeze(1)

        ode_func.set_context(h_n, h_l, degs)
        z_pred = integrate_ode(ode_func, z_s, torch.tensor([0.,1.],device=DEVICE)).cpu()

        zp_flat = MAN_CPU.logmap0(z_pred.float())
        zt_flat = MAN_CPU.logmap0(z_next.float())
        mse     = F.mse_loss(zp_flat, zt_flat).item()

        A_nf  = build_adj(graphs[s_next], nodes_snext).numpy()
        A_com = A_nf[np.ix_(iq, iq)]
        D     = MAN_CPU.dist(z_pred.unsqueeze(1), z_pred.unsqueeze(0)).numpy()
        P_    = 1.0/(np.exp((D - r4)/max(T4,0.05)) + 1.0)
        mask  = np.triu(np.ones_like(A_com, dtype=bool), k=1)
        try:
            auc = roc_auc_score(A_com[mask].astype(int), P_[mask])
        except: auc = 0.5
        within_results.append({"transition": f"{s}→{s_next}",
                                "MSE": mse, "AUC": auc})
        print(f"  {s}→{s_next} | MSE={mse:.5f} | AUC={auc:.4f}")

wr_df = pd.DataFrame(within_results)
print(f"\n  Mean within-training MSE : {wr_df['MSE'].mean():.5f}")
print(f"  Mean within-training AUC : {wr_df['AUC'].mean():.4f}")
print(f"  Held-out stage {HELD_OUT} MSE    : {res_ode['emb_mse']:.5f}")
print(f"  Held-out stage {HELD_OUT} AUC    : {res_ode['auc']:.4f}")


# ─────────────────────────────────────────────────────────────
# SECTION 13 — VISUALISATIONS (12 panels)
# ─────────────────────────────────────────────────────────────
print(f"\nSECTION 13: Generating Visualisations")

fig = plt.figure(figsize=(24, 20))
fig.suptitle(
    "Hyperbolic ODE Interpolator — C. elegans Connectome Development\n"
    "Hybrid mechanistic + neural ODE with lineage hyperedge features",
    fontsize=13, fontweight='bold', y=0.98
)

def disk(ax):
    ax.add_artist(plt.Circle((0,0),1,fill=False,color='black',lw=2))
    ax.set_xlim(-1.12,1.12); ax.set_ylim(-1.12,1.12); ax.set_aspect('equal')

# ── 1: Training loss ──────────────────────────────────────────
ax1 = fig.add_subplot(3,4,1)
ax1.plot(loss_history, 'steelblue', lw=1.5)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("ODE Training Loss", fontsize=10)
ax1.grid(alpha=0.4)

# ── 2: Poincaré disk — Stage 3 (start) ──────────────────────
ax2 = fig.add_subplot(3,4,2)
disk(ax2)
z3_np  = z3.numpy()
deg3v  = np.array([graphs[3].degree(v) for v in common_345])
# colour by lineage
lin_cols = np.array([{"ABa":"orange","ABp":"steelblue","MS":"green",
                       "C":"purple","P3":"brown"}.get(
                       neuron_lineage.get(v,"Unknown"),"gray")
                       for v in common_345])
ax2.scatter(z3_np[:,0], z3_np[:,1], c=lin_cols, s=15+2*deg3v, alpha=0.8)
ax2.set_title("Stage 3 (Start)\nColoured by lineage", fontsize=9)
for f, col in FOUNDERS.items():
    ax2.scatter([],[],c=col,s=40,label=f)
ax2.legend(fontsize=7, loc='lower right')

# ── 3: Predicted stage 4 (ODE) ───────────────────────────────
ax3 = fig.add_subplot(3,4,3)
disk(ax3)
z4o_np = z4_ode.numpy()
ax3.scatter(z4o_np[:,0], z4o_np[:,1], c=lin_cols, s=15+2*deg3v, alpha=0.8)
ax3.set_title(f"Predicted Stage {HELD_OUT} (ODE)\nColoured by lineage", fontsize=9)

# ── 4: True stage 4 ──────────────────────────────────────────
ax4 = fig.add_subplot(3,4,4)
disk(ax4)
z4t_np = z4_true.numpy()
deg4v  = np.array([graphs[HELD_OUT].degree(v) for v in common_345])
ax4.scatter(z4t_np[:,0], z4t_np[:,1], c=lin_cols, s=15+2*deg4v, alpha=0.8)
ax4.set_title(f"True Stage {HELD_OUT} (Trained)\nColoured by lineage", fontsize=9)

# ── 5: TP/FP/FN bar chart comparing 3 methods ────────────────
ax5 = fig.add_subplot(3,4,5)
cats   = ["TP","FP","FN"]
x      = np.arange(3); w = 0.25
for i, (res, lab, col) in enumerate([(res_geo,"Geodesic","steelblue"),
                                      (res_lin,"Linear","coral"),
                                      (res_ode,"ODE","limegreen")]):
    vals = [res[c] for c in cats]
    bars = ax5.bar(x + i*w, vals, w, label=lab, color=col, alpha=0.85)
    for b in bars:
        ax5.text(b.get_x()+b.get_width()/2, b.get_height()+3,
                 str(int(b.get_height())), ha='center', va='bottom', fontsize=7)
ax5.set_xticks(x+w); ax5.set_xticklabels(cats)
ax5.set_title(f"TP / FP / FN\nStage {HELD_OUT} Prediction", fontsize=10)
ax5.legend(fontsize=8); ax5.grid(axis='y', alpha=0.35)

# ── 6: AUC comparison bar ────────────────────────────────────
ax6 = fig.add_subplot(3,4,6)
methods = ["Geodesic","Linear","Hybrid ODE"]
aucs    = [res_geo["auc"], res_lin["auc"], res_ode["auc"]]
cols6   = ["steelblue","coral","limegreen"]
bars6   = ax6.bar(methods, aucs, color=cols6, alpha=0.85, edgecolor='black', width=0.5)
for b, v in zip(bars6, aucs):
    ax6.text(b.get_x()+b.get_width()/2, v+0.003, f"{v:.4f}",
             ha='center', va='bottom', fontweight='bold', fontsize=10)
ax6.axhline(0.5, color='red', ls='--', lw=1.5, label='Random')
ax6.set_ylim(0.4, 0.95)
ax6.set_ylabel("AUC"); ax6.legend(fontsize=8)
ax6.set_title("AUC Comparison", fontsize=10)
ax6.grid(axis='y', alpha=0.35)

# ── 7: F1 comparison ─────────────────────────────────────────
ax7 = fig.add_subplot(3,4,7)
f1s = [res_geo["f1"], res_lin["f1"], res_ode["f1"]]
bars7 = ax7.bar(methods, f1s, color=cols6, alpha=0.85, edgecolor='black', width=0.5)
for b, v in zip(bars7, f1s):
    ax7.text(b.get_x()+b.get_width()/2, v+0.003, f"{v:.4f}",
             ha='center', va='bottom', fontweight='bold', fontsize=10)
ax7.set_ylabel("F1 Score")
ax7.set_title("F1 Score Comparison", fontsize=10)
ax7.grid(axis='y', alpha=0.35)

# ── 8: Precision-Recall comparison ───────────────────────────
ax8 = fig.add_subplot(3,4,8)
for res, lab, col in [(res_geo,"Geodesic","steelblue"),
                       (res_lin,"Linear","coral"),
                       (res_ode,"ODE","limegreen")]:
    ax8.scatter(res["recall"], res["precision"], s=120,
                color=col, label=f"{lab}\n(P={res['precision']:.3f}, R={res['recall']:.3f})",
                edgecolors='black', zorder=5)
ax8.set_xlabel("Recall"); ax8.set_ylabel("Precision")
ax8.set_title("Precision vs Recall\n(Best F1 threshold)", fontsize=9)
ax8.legend(fontsize=7); ax8.grid(alpha=0.35)
ax8.set_xlim(0, 1); ax8.set_ylim(0, 1)

# ── 9: Embedding MSE comparison ──────────────────────────────
ax9 = fig.add_subplot(3,4,9)
mses = [res_geo["emb_mse"], res_lin["emb_mse"], res_ode["emb_mse"]]
bars9 = ax9.bar(methods, mses, color=cols6, alpha=0.85, edgecolor='black', width=0.5)
for b, v in zip(bars9, mses):
    ax9.text(b.get_x()+b.get_width()/2, v+1e-5, f"{v:.5f}",
             ha='center', va='bottom', fontweight='bold', fontsize=8)
ax9.set_ylabel("Embedding MSE (logmap₀ space)")
ax9.set_title("Embedding Quality\n(Lower = better)", fontsize=9)
ax9.grid(axis='y', alpha=0.35)

# ── 10: α balance over training (mechanistic vs neural) ───────
ax10 = fig.add_subplot(3,4,10)
# Plot predicted vs true position (scatter, common nodes)
pred_r  = z4_ode.norm(dim=1).numpy()
true_r  = z4_true.norm(dim=1).numpy()
ax10.scatter(true_r, pred_r, alpha=0.5, s=15, color='limegreen')
lims = [min(true_r.min(),pred_r.min()), max(true_r.max(),pred_r.max())]
ax10.plot(lims, lims, 'k--', lw=1.5, label='Perfect prediction')
corr = np.corrcoef(true_r, pred_r)[0,1]
ax10.set_xlabel("True Hyperbolic Radius")
ax10.set_ylabel("Predicted Hyperbolic Radius")
ax10.set_title(f"Radius Prediction Quality\nPearson r={corr:.3f}", fontsize=9)
ax10.legend(fontsize=8); ax10.grid(alpha=0.35)

# ── 11: Within-training AUC over transitions ──────────────────
ax11 = fig.add_subplot(3,4,11)
if within_results:
    trans  = [r["transition"] for r in within_results]
    w_aucs = [r["AUC"]        for r in within_results]
    cols11 = ["steelblue"]*len(trans)
    bars11 = ax11.bar(trans, w_aucs, color=cols11, alpha=0.85, edgecolor='black')
    ax11.axhline(res_ode["auc"], color='limegreen', ls='--', lw=2,
                 label=f'Held-out AUC={res_ode["auc"]:.3f}')
    ax11.axhline(0.5, color='red', ls=':', lw=1.5, label='Random')
    for b, v in zip(bars11, w_aucs):
        ax11.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.3f}",
                  ha='center', va='bottom', fontsize=8)
    ax11.set_ylabel("AUC"); ax11.set_ylim(0.4, 1.0)
    ax11.set_title("AUC per Transition\n(Blue=train, Green dashed=held-out)", fontsize=9)
    ax11.legend(fontsize=8); ax11.grid(axis='y', alpha=0.35)
    plt.setp(ax11.xaxis.get_majorticklabels(), rotation=30)

# ── 12: ODE trajectory for 10 sample neurons ─────────────────
ax12 = fig.add_subplot(3,4,12)
disk(ax12)
# Show actual ODE trajectory (intermediate steps)
ode_func.eval()
with torch.no_grad():
    samp_idx = list(range(0, min(15, len(common_345)), 1))
    z3_samp  = z3[samp_idx].to(DEVICE)
    h_n_s    = h_node3[samp_idx]
    h_l_s    = h_lin3[samp_idx]
    dg_s     = degs3[samp_idx]
    ode_func.set_context(h_n_s, h_l_s, dg_s)
    t_fine   = torch.linspace(0, 1, 20).to(DEVICE)
    z_traj   = odeint(ode_func, z3_samp, t_fine, method="rk4").cpu().numpy()
    # z_traj: (20, n_samp, 2)

cmap12 = cm.plasma(np.linspace(0, 1, len(samp_idx)))
for j, idx_s in enumerate(samp_idx):
    traj = z_traj[:, j, :]   # (20, 2)
    ax12.plot(traj[:, 0], traj[:, 1], '-', color=cmap12[j], lw=1, alpha=0.7)
    ax12.scatter(traj[0,0], traj[0,1], s=30, color=cmap12[j],
                 marker='>', zorder=5)
    ax12.scatter(traj[-1,0], traj[-1,1], s=30, color=cmap12[j],
                 marker='s', zorder=5)
ax12.scatter([],[],marker='>',c='gray',s=30,label='Stage 3 (start)')
ax12.scatter([],[],marker='s',c='gray',s=30,label='Pred. Stage 4 (end)')
ax12.legend(fontsize=7, loc='lower right')
ax12.set_title("ODE Trajectories\n(15 sample neurons, Stage 3→4)", fontsize=9)

plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig("ode_interpolator_results.png", dpi=150, bbox_inches='tight')
print("  Figure saved → ode_interpolator_results.png")
plt.show()

# ─────────────────────────────────────────────────────────────
# SECTION 14 — BIOLOGICAL INTERPRETATION
# ─────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("  BIOLOGICAL INTERPRETATION")
print("="*65)
print(f"""
Mechanistic/Neural balance:
  α = {torch.sigmoid(ode_func.alpha).item():.3f}
  (α=1 → pure mechanistic radial ODE,  α=0 → pure neural)

  If α > 0.5: the radial ODE captures most of the dynamics.
              Development is largely driven by degree-dependent
              centripetal forces (hubs pulled to center).
  If α < 0.5: the neural component dominates.
              Development involves angular and collective motion
              beyond what the radial model captures.

Lineage hyperedge contribution:
  The ODE takes lin_dir and lin_dist as inputs — encoding the
  "pull" of each neuron toward its lineage centroid.
  If the model learned to USE these features (visible from
  gradient magnitudes), it confirms that neurons move coherently
  with their lineage family — your mentor's core hypothesis.

TP / FP / FN interpretation for stage {HELD_OUT} prediction:
  TP={res_ode["TP"]:4d} — synapses correctly predicted to form
  FP={res_ode["FP"]:4d} — geometrically close pairs that didn't wire
              (potential synapses suppressed by biology)
  FN={res_ode["FN"]:4d} — real synapses missed
              (wiring driven by factors outside our model)

  Precision={res_ode['precision']:.3f}: out of predicted edges, this fraction are real.
  Recall   ={res_ode['recall']:.3f}:    out of real edges, this fraction we found.
""")

Device: cuda
SECTION 1: Loading Data
----------------------------------------
  Lineage tree: 1203 cells
  Stage 1: 187 nodes, 782 edges
  Stage 2: 194 nodes, 995 edges
  Stage 3: 198 nodes, 990 edges
  Stage 4: 204 nodes, 1193 edges
  Stage 5: 211 nodes, 1561 edges
  Stage 6: 216 nodes, 1540 edges
  Stage 7: 222 nodes, 2133 edges
  Stage 8: 219 nodes, 2061 edges

SECTION 2: Lineage Assignment
----------------------------------------
  Lineage assignments:
    ABa  :  96
    ABp  :  82
    MS   :   5
    E    :   0
    C    :   2
    P3   :   0
    Unknown:  40

SECTION 6: Pre-training GAT
  Held-out stage: 4
  Training stages: [1, 2, 3, 5, 6, 7, 8]
  GAT ep   0  loss 6.4251
  GAT ep 100  loss 1.4575
  GAT ep 200  loss 1.4199
  GAT ep 300  loss 1.3874
  GAT ep 400  loss 1.3117
  GAT ep 500  loss 1.2886
  GAT ep 600  loss 1.2296
  GAT frozen.

Training hyperbolic embeddings …
  Stage 1 … done | r=0.277 T=0.246
  Stage 2 … done | r=0.643 T=0.378
  Stage 3 … done | r=0.932 T=0.507
  Stage

NameError: name 'transitions' is not defined